# ArcGIS API for Python: Pure Referenced Image Collection Publishing

This notebook demonstrates how to create a registered S3 cloud storage **Image Collection**  **by reference** using the **ArcGIS API for Python** (no local `arcpy` and no geoprocessing toolbox imports required).

### Step 1: Connect and Authenticate to ArcGIS Enterprise Portal

In [ ]:
from os import getenv
from dotenv import load_dotenv

from arcgis.gis import GIS
from arcgis.raster.analytics import create_image_collection, get_datastores

load_dotenv()

portal_url = getenv("PORTAL_URL")
client_id = getenv("CLIENT_ID")

gis = GIS(url=portal_url, client_id=client_id)
print(f"Hello, {gis.users.me.username}")

### Step 2: Locate and Verify existing S3 Cloud Datastore

We can use the Raster Analytics `get_datastores` helper function to query registered data stores.

In [ ]:
cloudstore_name = getenv("CLOUDSTORE_NAME")

datastores = get_datastores(gis)
cloud_datastores = datastores.search(types="cloudStore")

matching_cloudstore = [ds for ds in cloud_datastores if cloudstore_name == ds.datapath][0]
if not matching_cloudstore:
    raise ValueError(f"No cloud datastore named '{cloudstore_name}' found in datastores.")

print(f"Found cloud datastore: {matching_cloudstore}")

### Step 3: Publish Referenced Image Collection

We create an empty Image Service to act as a placeholder with the proper `create_params` (`copyData: false` and `esriImageServiceSourceTypeMosaicDataset`). We then pass this Item to **`create_image_collection`** to populate the dataset by reference.

In [ ]:
service_name = "raster_collection"
raster_files = ["test.tif"] 

# 1. Format the input referenced datasets as a list of strings
input_rasters = [f"{matching_cloudstore.datapath}/{f.lstrip('/')}" for f in raster_files]
print(f"Referencing data... '{input_rasters}'")

# 2. Define the image service properties
create_params = {
    "name": service_name,
    "capabilities": "Image, Catalog, Mensuration, Metadata", # Key
    "cacheControlMaxAge": "43200",
    "provider": "ArcObjectsRasterRendering",
    "properties": {
        # "path": N/A, the Image Collection will handle the path to the underlying rasters
        "isManaged": True,
        "isCached": False,
        "esriImageServiceSourceType": "esriImageServiceSourceTypeMosaicDataset",
        "isTiledImagery": "false",
        "colormapToRGB": "false",
        "description": f"Referenced Image Collection dynamically published by reference from Cloud Datastore '{matching_cloudstore.datapath}'.",
        "defaultResamplingMethod": 1
    },
    "copyData": False
}

# 3. Create empty image service item
# This step will circumvent a direct create_image_collection() function creating an Image Service Item with the (hosted) flag
print(f"\nCreating placehoder Item... '{service_name}'")
empty_item = gis.content.create_service(
    name=service_name,
    service_type="imageService",
    create_params=create_params,
    description=f"Referenced Image Collection dynamically published by reference from Cloud Datastore '{matching_cloudstore.datapath}'.",
    tags=["Image Service", "RasterCollection"]
)

# 4. Publish Image Collection into empty image service item
print(f"\nPublishing Image Collection... '{service_name}'")
published_item = create_image_collection(
    image_collection=empty_item,
    input_rasters=input_rasters,
    raster_type_name="Raster Dataset",
    context={"byref": True},
    gis=gis
)

print("\nSuccess!")
print("Item Title:", published_item.title)
print("Item ID:", published_item.id)
print("Service URL:", getattr(published_item, "url", "N/A"))
print("Hosted Status:", "Hosted" in published_item.typeKeywords)